## This notebook provides the template to run post hoc explainers including GNNExplainer, PGExplainer on the E3 dataset.

## You can choose to download the data yourself using the urls mentioned in this notebook or mount this notebook to the data directory in Beryl to run it automatically. The directory information is present in the repository Readme.

In [ ]:
'''
Importing the require libraries here
'''
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import torch
from torch_geometric.data import Data
import os
import torch.nn.functional as F
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
warnings.filterwarnings('ignore')
from torch_geometric.loader import NeighborLoader
import multiprocessing

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

In [ ]:
import os
'''
Setting working directory for rivana jupyter notebook
'''
os.chdir("content")
%pwd

In [ ]:
from pprint import pprint
import gzip
from sklearn.manifold import TSNE
import json
import copy
import os

In [ ]:
'''
This is the main featurizer. It constructs the graph for the cadets dataset.

Args:
    df (DataFrame): This is the main dataframe containing all the system events from the cadets dataset.

return:
    features (list): Contains word2vec encoded feature vectors for each node
    feat_labels (list): Contains label for each node
    edge_index (list): Contains information about edges between nodes in the graph.
    mapp (list): contains id of each node
'''

tokens = ['SUBJECT_PROCESS',
          'FILE_OBJECT_FILE',
          'NetFlowObject'
         ]

def prepare_graph(df):
    nodes = {}
    labels = {}
    edges = []
    proc = {}
    
    global tokens
    dummies = {token: index for index, token in enumerate(tokens)}


    for i in range(len(df)):
        x = df.iloc[i]
        action = x["action"]
        
        actorid = x["actorID"]
        if not (actorid in nodes):
            nodes[actorid] =  []
        nodes[actorid].append(x['exec'])
        nodes[actorid].append(action)
        if x['path'] != '':
            nodes[actorid].append(x['path'])
        labels[actorid] = dummies[x['actor_type']]

        objectid = x["objectID"]
        if not (objectid in nodes):
            nodes[objectid] =  []
        nodes[objectid].append(x['exec'])
        nodes[objectid].append(action)
        if x['path'] != '':
             nodes[objectid].append(x['path'])
        labels[objectid] = dummies[x['object']]

        edges.append(( actorid, objectid, x['timestamp'] ))
        
        proc[actorid] = x['actorID']

    features = []
    feat_labels = []
    edge_index = [[],[]]
    index  = {}
    edges_timestamps = []
    mapp = []

    all_procs = set()

    for k,v in nodes.items():
        features.append(v)
        feat_labels.append(labels[k])
        index[k] = len(features) - 1
        mapp.append(k)
        
        if k in proc:
            all_procs.add(proc[k])

    for x in edges:
        src = index[x[0]]
        dst = index[x[1]]

        edge_index[0].append(src)
        edge_index[1].append(dst)
        edges_timestamps.append(x[2])

    idx_to_proc = {}
    for i in range(len(mapp)):
        if mapp[i] in proc:
            idx_to_proc[i] = proc[mapp[i]]
            
    all_procs = list(all_procs)

    return features,feat_labels,edge_index,mapp

In [ ]:
from torch_geometric.nn import GCNConv
from torch_geometric.nn import SAGEConv, GATConv
import torch.nn as nn

'''
Defining the model. The model consists mainly of graph sage and graph attention layers
'''
class GCN(torch.nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.conv1 = SAGEConv(in_channel, 32, normalize=True)
        self.conv2 = SAGEConv(32, out_channel, normalize=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        return x

In [ ]:
from gensim.models.callbacks import CallbackAny2Vec
import gensim
from gensim.models import Word2Vec
from multiprocessing import Pool
from itertools import compress
from tqdm import tqdm
import time

class EpochSaver(CallbackAny2Vec):
    '''Callback to save model after each epoch.'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        model.save('word2vec_theia_E3.model')
        self.epoch += 1

In [ ]:
class EpochLogger(CallbackAny2Vec):
    '''Callback to log information about training'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_begin(self, model):
        print("Epoch #{} start".format(self.epoch))

    def on_epoch_end(self, model):
        print("Epoch #{} end".format(self.epoch))
        self.epoch += 1

In [ ]:
logger = EpochLogger()
saver = EpochSaver()

In [ ]:
f = open("darpatc/theia_train.txt")
data = f.read().split('\n')
data = [line.split('\t') for line in data]
df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
df = df.dropna()
df.sort_values(by='timestamp', ascending=True,inplace=True)
df = df[df['actor_type'] == 'SUBJECT_PROCESS'] 
df = df[df['object'].isin(tokens)]    

In [ ]:
def add_attributes(d,p):
    
    f = open(p)
    data = [json.loads(x) for x in f if "EVENT" in x]

    info = []
    for x in data:
        try:
            action = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['type']
        except:
            action = ''
        try:
            actor = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['subject']['com.bbn.tc.schema.avro.cdm18.UUID']
        except:
            actor = ''
        try:
            obj = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObject']['com.bbn.tc.schema.avro.cdm18.UUID']
        except:
            obj = ''
        try:
            timestamp = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['timestampNanos']
        except:
            timestamp = ''
        try:
            cmd = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['properties']['map']['cmdLine']
        except:
            cmd = ''
        try:
            path = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObjectPath']['string']
        except:
            path = ''
        try:
            path2 = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObject2Path']['string']
        except:
            path2 = ''
        try:
            obj2 = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObject2']['com.bbn.tc.schema.avro.cdm18.UUID']
            info.append({'actorID':actor,'objectID':obj2,'action':action,'timestamp':timestamp,'exec':cmd, 'path':path2})
        except:
            pass

        info.append({'actorID':actor,'objectID':obj,'action':action,'timestamp':timestamp,'exec':cmd, 'path':path})

    rdf = pd.DataFrame.from_records(info).astype(str)
    d = d.astype(str)

    return d.merge(rdf,how='inner',on=['actorID','objectID','action','timestamp']).drop_duplicates()

In [ ]:
p = "ta1-cadets-e3-official.json"
f = open(p)
data = [json.loads(x) for x in f if "EVENT" in x]
data[0]

In [ ]:
p = "ta1-cadets-e3-official.json"
f = open(p)
data = [json.loads(x) for x in f if "EVENT" in x]
data[0]

In [ ]:
data[0]

In [ ]:
df = add_attributes(df,"ta1-theia-e3-official-1r.json")

In [ ]:
phrases,labels,edges,mapp = prepare_graph(df)

In [ ]:
#word2vec = Word2Vec(sentences=phrases, vector_size=30, window=5, min_count=1, workers=8,epochs=300,callbacks=[saver,logger])

In [ ]:
'''
Defining the train and test function in this cell 
'''
from sklearn.utils import class_weight
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

model = GCN(30,5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [ ]:
import math
class PositionalEncoding():

    def __init__(self, d_model ,max_len = 100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:,0::2] = torch.sin(position * div_term)
        self.pe[:,1::2] = torch.cos(position * div_term)

    def embed(self, x):
        x = x + self.pe[:x.size(0)]
        return x

encoder = PositionalEncoding(30)

In [ ]:
from collections import Counter
word2vec = Word2Vec.load("word2vec_theia_E3.model")

def infer(doc):  
  word_emb = []
  for word in doc:
    if word in word2vec.wv:
      word_emb.append(word2vec.wv[word])
  
  if len(word_emb) == 0:
    return np.zeros(30)

  out_emb = torch.tensor(word_emb,dtype=torch.float)
  if len(doc) < 100000:
    out_emb = encoder.embed(out_emb)
  out_emb = out_emb.detach().cpu().numpy()
  out_emb = np.mean(out_emb,axis=0)
  return out_emb

In [ ]:
from torch_geometric import utils

################################## Training Main Model #####################################

criterion = CrossEntropyLoss()

nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)  

graph = Data(x=torch.tensor(nodes,dtype=torch.float).to(device),y=torch.tensor(labels,dtype=torch.long).to(device), edge_index=torch.tensor(edges,dtype=torch.long).to(device))
graph.n_id = torch.arange(graph.num_nodes)
mask = torch.tensor([True]*graph.num_nodes, dtype=torch.bool)

for m_n in range(1):

  loader = NeighborLoader(graph, num_neighbors=[-1,-1], batch_size=5000,input_nodes=mask)
  total_loss = 0
  for subg in loader:
      model.train()
      optimizer.zero_grad() 
      out = model(subg.x, subg.edge_index) 
      loss = criterion(out, subg.y) 
      loss.backward() 
      optimizer.step()      
      total_loss += loss.item() * subg.batch_size
  print(total_loss / mask.sum().item())

  loader = NeighborLoader(graph, num_neighbors=[-1,-1], batch_size=5000,input_nodes=mask)
  for subg in loader:
      model.eval()
      out = model(subg.x, subg.edge_index)

      sorted, indices = out.sort(dim=1,descending=True)
      conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
      conf = (conf - conf.min()) / conf.max()
    
      pred = indices[:,0]
      cond = (pred == subg.y) | (conf >= 0.9)
      mask[subg.n_id[cond]] = False
      
  torch.save(model.state_dict(), f'lword2vec_gnn_theia{m_n}_E3.pth')
  print(f'Model# {m_n}. {mask.sum().item()} nodes still misclassified \n')

In [ ]:
from itertools import compress
from torch_geometric import utils

def construct_neighborhood(ids,mapp,edges,hops):
    if hops == 0:
        return set()
    else:
        neighbors = set()
        for i in range(len(edges[0])):
            if mapp[edges[0][i]] in ids:
                neighbors.add(mapp[edges[1][i]])
            if mapp[edges[1][i]] in ids:
                neighbors.add(mapp[edges[0][i]])
        return neighbors.union( construct_neighborhood(neighbors,mapp,edges,hops-1) )

In [ ]:
def helper(MP,all_pids,GP,edges,mapp):

    TP = MP.intersection(GP)  
    FP = MP - GP              
    FN = GP - MP              
    TN = all_pids - (GP | MP)
    
    two_hop_gp = construct_neighborhood(GP,mapp,edges,2)
    two_hop_tp = construct_neighborhood(TP,mapp,edges,2)
    FPL = FP - two_hop_gp
    TPL = TP.union(FN.intersection(two_hop_tp))
    FN = FN - two_hop_tp
    
    alerts = TP.union(FP)

    TP,FP,FN,TN = len(TPL),len(FPL),len(FN),len(TN)
    
    FPR = FP / (FP+TN)
    TPR = TP / (TP+FN)

    print(f"Number of True Positives: {TP}")
    print(f"Number of Fasle Positives: {FP}")
    print(f"Number of False Negatives: {FN}")

    prec = TP / (TP + FP)
    print(f"Precision: {prec}")

    rec = TP / (TP + FN)
    print(f"Recall: {rec}")

    fscore = (2*prec*rec) / (prec + rec)
    print(f"Fscore: {fscore}\n")
    
    #return alerts
    return TPL,FPL

In [ ]:
f = open("darpatc/theia_test.txt")
data = f.read().split('\n')
data = [line.split('\t') for line in data]
df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
df = df.dropna()
df.sort_values(by='timestamp', ascending=True,inplace=True)

In [ ]:
df = add_attributes(df,"ta1-theia-e3-official-6r.json.8")

In [ ]:
df = df[df['actor_type'] == 'SUBJECT_PROCESS'] 
df = df[df['object'].isin(tokens)]    

In [ ]:
gt = open("theia.txt").read()
GT_mal = set(gt.split("\n"))

data = df

phrases,labels,edges,mapp = prepare_graph(data)
nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)  

all_ids = list(data['actorID']) + list(data['objectID'])
all_ids = set(all_ids)

In [ ]:
graph = Data(x=torch.tensor(nodes,dtype=torch.float).to(device),y=torch.tensor(labels,dtype=torch.long).to(device), edge_index=torch.tensor(edges,dtype=torch.long).to(device))
graph.n_id = torch.arange(graph.num_nodes)
flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool)

for m_n in range(1):
  model.load_state_dict(torch.load(f'lword2vec_gnn_theia{m_n}_E3.pth',map_location=torch.device('cpu')))
  loader = NeighborLoader(graph, num_neighbors=[-1,-1], batch_size=5000)    
  for subg in loader:
      model.eval()
      out = model(subg.x, subg.edge_index)

      sorted, indices = out.sort(dim=1,descending=True)
      conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
      conf = (conf - conf.min()) / conf.max()
    
      pred = indices[:,0]
      cond = (pred == subg.y)# & (conf > 0.51)
      flag[subg.n_id[cond]] = torch.logical_and(flag[subg.n_id[cond]], torch.tensor([False]*len(flag[subg.n_id[cond]]), dtype=torch.bool))

index = utils.mask_to_index(flag).tolist()
ids = set([mapp[x] for x in index])
metrics = helper(set(ids),set(all_ids),GT_mal,edges,mapp) 

In [ ]:
graph.node_ids = mapp
TPL = metrics[0]

process_alerts = set()
for i in range(len(mapp)):
    if mapp[i] in TPL and labels[i] == 0:
        process_alerts.add(mapp[i])

node_indices = []
for i in range(len(graph.node_ids)):
    if graph.node_ids[i] in process_alerts:
        node_indices.append(i)

In [ ]:
gt = ['8B161C29-0000-0000-0000-000000000020', '5418F32A-0000-0000-0000-000000000020', '4E18B22A-0000-0000-0000-000000000020', '5518F72A-0000-0000-0000-000000000020', '6B18B82B-0000-0000-0000-000000000020', '5618FB2A-0000-0000-0000-000000000020', '5818FF2A-0000-0000-0000-000000000020', '64189C2B-0000-0000-0000-000000000020', '65189F2B-0000-0000-0000-000000000020', '6618A32B-0000-0000-0000-000000000020', '6718A92B-0000-0000-0000-000000000020', '4F18B22A-0000-0000-0000-000000000020', '5318F12A-0000-0000-0000-000000000020', '06187E2A-0000-0000-0000-000000000020', '78188B2C-0000-0000-0000-000000000020', '6818A92B-0000-0000-0000-000000000020', '79188C2C-0000-0000-0000-000000000020', '7B18A02C-0000-0000-0000-000000000020', '7D18A12C-0000-0000-0000-000000000020', '7E18A32C-0000-0000-0000-000000000020', '7F18A52C-0000-0000-0000-000000000020', '8018A52C-0000-0000-0000-000000000020', '8118AB2C-0000-0000-0000-000000000020', '8318AC2C-0000-0000-0000-000000000020', 'CD186B2D-0000-0000-0000-000000000020', 'CE186C2D-0000-0000-0000-000000000020', '8718032D-0000-0000-0000-000000000020', 'D0187D2D-0000-0000-0000-000000000020', 'D1187E2D-0000-0000-0000-000000000020', 'D518C12D-0000-0000-0000-000000000020', 'D618C22D-0000-0000-0000-000000000020', '6918A92B-0000-0000-0000-000000000020', '6D18C82B-0000-0000-0000-000000000020', 'FA1F9E30-0000-0000-0000-000000000020', 'C21F0F30-0000-0000-0000-000000000020', 'EF1F3230-0000-0000-0000-000000000020', '1920A530-0000-0000-0000-000000000020', 'EB1F2430-0000-0000-0000-000000000020', 'C31F0F30-0000-0000-0000-000000000020', 'EE1F3230-0000-0000-0000-000000000020', 'F71F9730-0000-0000-0000-000000000020', '7F1FD92F-0000-0000-0000-000000000020', '54387BBE-0200-0000-0000-000000000020', 'FB1F9E30-0000-0000-0000-000000000020', '61739171-0100-0000-0000-000000000020', '55387DBE-0200-0000-0000-000000000020', '60739071-0100-0000-0000-000000000020', '223838BC-0200-0000-0000-000000000020', '23383FBC-0200-0000-0000-000000000020', '26200C31-0000-0000-0000-000000000020', '07369EB8-0200-0000-0000-000000000020', '29202531-0000-0000-0000-000000000020', '0836A1B8-0200-0000-0000-000000000020', '0A36B3B8-0200-0000-0000-000000000020', 'C43590B6-0200-0000-0000-000000000020', 'EB3597B7-0200-0000-0000-000000000020', 'EC3598B7-0200-0000-0000-000000000020', 'ED35A9B7-0200-0000-0000-000000000020', '980A5F85-0200-0000-0000-000000000020', 'EE35ABB7-0200-0000-0000-000000000020', 'D037D3BA-0200-0000-0000-000000000020', 'D137D4BA-0200-0000-0000-000000000020', 'D237D6BA-0200-0000-0000-000000000020', '273847BC-0200-0000-0000-000000000020', 'FC1F9E30-0000-0000-0000-000000000020', '1A20AE30-0000-0000-0000-000000000020', '1B20AF30-0000-0000-0000-000000000020', '32220A33-0000-0000-0000-000000000020', '6D72B270-0100-0000-0000-000000000020', '6E72B670-0100-0000-0000-000000000020', 'BC72F370-0100-0000-0000-000000000020', 'BD72F470-0100-0000-0000-000000000020', 'BE72F470-0100-0000-0000-000000000020', 'BF72F470-0100-0000-0000-000000000020', 'C072F470-0100-0000-0000-000000000020', 'C172F570-0100-0000-0000-000000000020', 'C272F570-0100-0000-0000-000000000020', '960A5385-0200-0000-0000-000000000020', '970A5485-0200-0000-0000-000000000020', '7135AAB4-0200-0000-0000-000000000020', '1D38E3BB-0200-0000-0000-000000000020', '7235B0B4-0200-0000-0000-000000000020', '3A3530B4-0200-0000-0000-000000000020', '01368CB8-0200-0000-0000-000000000020', 'B53501B6-0200-0000-0000-000000000020', 'BC353DB6-0200-0000-0000-000000000020', '033690B8-0200-0000-0000-000000000020', 'BD353EB6-0200-0000-0000-000000000020', '043694B8-0200-0000-0000-000000000020', '8F3745BA-0200-0000-0000-000000000020', '903745BA-0200-0000-0000-000000000020', 'C1357AB6-0200-0000-0000-000000000020', 'C2357BB6-0200-0000-0000-000000000020', 'C6359FB6-0200-0000-0000-000000000020', 'C735BAB6-0200-0000-0000-000000000020', 'C835BAB6-0200-0000-0000-000000000020', '3F36C5B8-0200-0000-0000-000000000020', '4036C6B8-0200-0000-0000-000000000020', 'CA37B9BA-0200-0000-0000-000000000020', 'CB37BFBA-0200-0000-0000-000000000020', 'C337B3BA-0200-0000-0000-000000000020', 'C437B3BA-0200-0000-0000-000000000020', '7737F1B9-0200-0000-0000-000000000020', '7837F1B9-0200-0000-0000-000000000020', 'CD37C5BA-0200-0000-0000-000000000020', 'CE37C9BA-0200-0000-0000-000000000020', 'B33501B6-0200-0000-0000-000000000020', 'B43501B6-0200-0000-0000-000000000020', '3A3530B4-0200-0000-0000-000000000020', '5336F2B8-0200-0000-0000-000000000020', '55361BB9-0200-0000-0000-000000000020', '56361CB9-0200-0000-0000-000000000020', 'F135CDB7-0200-0000-0000-000000000020', 'C935BBB6-0200-0000-0000-000000000020', 'CA35BBB6-0200-0000-0000-000000000020', 'F335D6B7-0200-0000-0000-000000000020', 'F235D6B7-0200-0000-0000-000000000020', 'B63501B6-0200-0000-0000-000000000020', '9F376ABA-0200-0000-0000-000000000020', 'B73501B6-0200-0000-0000-000000000020', '3536B9B8-0200-0000-0000-000000000020', 'A1377EBA-0200-0000-0000-000000000020', '283847BC-0200-0000-0000-000000000020', '293847BC-0200-0000-0000-000000000020', '4D38FABD-0200-0000-0000-000000000020', '62175519-0400-0000-0000-000000000020', '1D173C19-0400-0000-0000-000000000020']
gt_coverage = set(gt)

# GNNExplainer

In [ ]:
from torch_geometric.explain import Explainer, PGExplainer, ModelConfig, GNNExplainer
from torch_geometric.utils import k_hop_subgraph
import torch

all_explainer_edges = []
for node_idx in node_indices:
    
    explainer = Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=200),
        explanation_type='model',
        node_mask_type='attributes',
        edge_mask_type='object',
        model_config=dict(
            mode='multiclass_classification',
            task_level='node',
            return_type='log_probs',
        ),
    )
    
    explanation = explainer(
        x=graph.x,
        edge_index=graph.edge_index,
        target=graph.y,
        index=node_idx,
    )

    explainer_edges = explanation.edge_mask
    indices = (explainer_edges > 0).nonzero(as_tuple=True)[0].tolist()
    indices = set(indices)
    all_explainer_edges = all_explainer_edges + list(indices)
    
    #subset, edge_index, mapping, hard_edge_mask = k_hop_subgraph(
    #    node_idx, num_hops=2, edge_index=graph.edge_index, relabel_nodes=True
    #)
    #subset_indices = subset.cpu().numpy()
    #node_labels = [graph.node_ids[i] for i in subset_indices]

gt_edge_indices = set()
for i in range(graph.edge_index.shape[1]):
    src, dst = graph.edge_index[0][i], graph.edge_index[1][i]
    if graph.node_ids[src] in gt_coverage and graph.node_ids[dst] in gt_coverage: 
        gt_edge_indices.add(i) 

all_explainer_edges = set(all_explainer_edges)
gt_edge_indices = set(gt_edge_indices)  

In [ ]:
tp = len(all_explainer_edges.intersection(gt_edge_indices))
fp = len(all_explainer_edges - gt_edge_indices)
fn = len(gt_edge_indices - all_explainer_edges)

N = len(graph.edge_index[0])
total_edges = set(range(N))  
tn = len(total_edges - all_explainer_edges - gt_edge_indices)

# Print metrics
print(f'True Positives (TP): {tp}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')
print(f'True Negatives (TN): {tn}')

# PGExplainer

In [ ]:
from torch_geometric.explain import Explainer, PGExplainer, ModelConfig, GNNExplainer
from torch_geometric.utils import k_hop_subgraph
import torch

all_explainer_edges_pg = []
for node_idx in node_indices:
    
    explainer = Explainer(
        model=model,
        algorithm=PGExplainer(epochs=30, lr=0.003),
        explanation_type='phenomenon',
        edge_mask_type='object',
        model_config=ModelConfig(
            mode='multiclass_classification',
            task_level='node',
            return_type='log_probs'
        ),
    )
    
    # Move the PGExplainer's MLP to the CUDA device
    explainer.algorithm.mlp.to(device)
    
    for epoch in range(30):
        for index in [node_idx]: 
            loss = explainer.algorithm.train(
                epoch,
                model.to(device),
                graph.x.to(device),
                graph.edge_index.to(device),
                target=graph.y.to(device),
                index=index
            )
    
    explanation = explainer(
        x=graph.x.to(device),
        edge_index=graph.edge_index.to(device),
        target=graph.y.to(device),
        index=node_idx,
    )
    
    explainer_edges = explanation.edge_mask
    indices = (explainer_edges > 0).nonzero(as_tuple=True)[0].tolist()
    indices = set(indices)
    all_explainer_edges_pg.extend(indices)

    #subset, edge_index, mapping, hard_edge_mask = k_hop_subgraph(
    #    node_idx, num_hops=2, edge_index=graph.edge_index, relabel_nodes=True
    #)
    #subset_indices = subset.cpu().numpy()
    #node_labels = [graph.node_ids[i] for i in subset_indices]

gt_edge_indices = set()
for i in range(graph.edge_index.shape[1]):
    src, dst = graph.edge_index[0][i], graph.edge_index[1][i]
    if graph.node_ids[src] in gt_coverage and graph.node_ids[dst] in gt_coverage: 
        gt_edge_indices.add(i) 

all_explainer_edges = set(all_explainer_edges_pg)
gt_edge_indices = set(gt_edge_indices)  

tp = len(all_explainer_edges.intersection(gt_edge_indices))
fp = len(all_explainer_edges - gt_edge_indices)
fn = len(gt_edge_indices - all_explainer_edges)

N = len(graph.edge_index[0])
total_edges = set(range(N))  
tn = len(total_edges - all_explainer_edges - gt_edge_indices)

# Print metrics
print(f'True Positives (TP): {tp}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')
print(f'True Negatives (TN): {tn} \n')

In [ ]:
# Print metrics
print(f'True Positives (TP): {tp}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')
print(f'True Negatives (TN): {tn} \n')